# DePlot — DIMER E2E chart-to-table fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/deplot-chart-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/deplot-chart-pipeline/blob/main/tutorials/deplot_chart_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Fdeplot-ffcc4d?style=flat)](https://huggingface.co/google/deplot) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fpix2struct-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/pix2struct) [![arXiv](https://img.shields.io/badge/arXiv-2212.10505-b31b1b.svg)](https://arxiv.org/abs/2212.10505)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** chart-to-table extraction and bounded supervised fine-tuning of the decoder's last blocks on a chart/table dataset, using the pinned `google/deplot` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/deplot_chart_pipeline/`, at revision `8d2ed5e52927`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `6e76d62430da16986be3426bae32301fb9115397` (~1133 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `google/deplot` snapshot (a 1.13 GB `model.safetensors`), downloads one digest-pinned parquet shard of SynthChartNet charts from the Hugging Face Hub (516 MB, no credential, refused on any size or SHA-256 mismatch), converts each chart's OTSL table into DePlot's own linearised format, draws a seeded chart-type-stratified sample of 360 training, 80 validation and 160 test charts with no image shared between splits, extracts the table of a drawn bar chart through the inference contract with an input manifest and a rejection probe, scores the frozen model on the test charts beside three non-neural baselines, runs a bounded fine-tuning of the decoder's last blocks with validation-cell-accuracy epoch selection, scores the held-out charts again per chart type, re-extracts the drawn chart with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify table parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). A CUDA runtime is used automatically when present; the CPU path works but is slow (every chart is encoded at up to 2,048 patches and its table generated token by token), and the timings of the first clean run are recorded in `docs/release-verification.md`.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip holding a `records.jsonl` (or `records.json`) of objects with `id`, `image` (a file name inside the zip), `target_text` (the chart's table in DePlot's linearised format: `TITLE | <title>` then rows joined by ` <0x0A> ` and cells by ` | `) and optional `image_id` and `chart_type`, beside the image files. They pass through the same validation, seeded image-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the SynthChartNet sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

`google/deplot` is the DePlot model of Liu et al. (2022) — a Pix2Struct image encoder over variable-resolution 16×16 patches (up to 2,048 per image) and a 12-layer text decoder, 282,285,696 parameters, fine-tuned by Google Research to translate a plot into its underlying data table — published under the **Apache-2.0** licence. The fixed instruction is **rendered as a text header above the chart** (the Pix2Struct convention, in Pillow's bundled font so no font is downloaded), the composite is encoded, and the decoder generates a linearised table with greedy decoding under a caller-owned `max_new_tokens` budget: a `TITLE | …` row, then rows joined by the literal token `<0x0A>` and cells by `|`. **No score exists**: the table is generated text with no probability and no correctness signal, and a well-formed table is **not evidence that its numbers** were read from the chart.

What this notebook adds to inference is **adaptation with labelled tables**. The dataset is real chart/table pairs from a chart family DePlot was not fine-tuned on: SynthChartNet (docling-project), bar, pie, stacked-bar and line charts rendered with Matplotlib, Seaborn and Pyecharts from financial-report tables, published under the **CDLA-Permissive-2.0** licence. The notebook downloads **one pinned parquet shard** (516 MB, SHA-256 pinned in the carried module) and draws a seeded, chart-type-stratified subset from it. **Target format:** SynthChartNet stores each table as OTSL, with the axis categories of a bar, pie or stacked-bar chart along its first row; the frozen checkpoint, run on charts of this shard, writes one row per category instead (and a line chart's x points as rows, as OTSL already does), so the carried converter transposes bar, pie and stacked-bar tables, keeps line tables, and writes them in the model's own `TITLE |` / ` <0x0A> ` / ` | ` convention. The honest question is narrow: does a bounded adaptation of the decoder's last blocks on a few hundred charts of a new family move held-out cell accuracy at all, and on which chart type?

Scores are **relaxed position-wise cell accuracy** (the repository's metric: every expected cell compared with the predicted cell at the same position, numbers within 5 %), **RNSS** (the relative number set similarity of the ChartQA and DePlot papers, which ignores layout) and exact-table match, overall and per chart type. Three references frame them: the **empty baseline**, the **header-only baseline** and the **medoid baseline** — systems that never look at the chart. Nothing here is a quality claim about your charts: it is one seeded split of one shard.

**Weight-format note:** the pinned revision ships the model as SafeTensors (`model.safetensors`, digest-pinned in the manifest, loaded in float32); the processor is the VQA variant, which renders the header. Section 3 stages and digest-verifies the snapshot before the processor or the model is constructed.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, metrics and dataset modules guarantee; stage and digest-verify the immutable upstream snapshot; download a digest-pinned shard of chart/table pairs, convert its tables to the model's own output convention, validate them and split them by chart type without sharing an image; extract a table through the public API on a drawn chart and read `text`, the parsed `table`, `new_tokens` and `truncated` correctly (generated text, no score); score the frozen model's cell accuracy and RNSS beside three non-neural baselines and read the per-chart-type breakdown; run a bounded fine-tuning with explicit hyperparameters and validation-based epoch selection; evaluate on a held-out test split; re-extract a drawing from a different image family with the adapted model; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** chart question answering or reasoning over the extracted table (the DePlot paper pairs the table with a language model; none is bundled), charts in images with several plots or a plot embedded in a page (one chart image per call), any instruction other than the fixed plot-to-table prompt, batch throughput, sampling or beam search, the DePlot paper's relative mapping similarity (RMS) and evaluation on ChartQA or PlotQA, fine-tuning of the image encoder, the embeddings or the output projection, training on charts that are not the pinned sample or your own uploads, and any claim that a SynthChartNet split stands in for your charts. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available; a GPU runtime is recommended for Sections 6–8. Every chart is encoded at up to 2,048 patches and its table generated token by token, so each extraction costs seconds on CPU. The pinned `torch==2.14.0` install, the 1.13 GB checkpoint and the 516 MB chart shard are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what an encoder–decoder model's generated tokens are; how a chart's data table is laid out (one row per category, one column per series); that a well-formed table is not a correct one; what a score against baselines that never see the image does and does not show.
- **Data contract:** records carry `id`, `image` and `target_text` — an image file decodable by Pillow with sides between `MIN_IMAGE_SIDE` (16) and `MAX_IMAGE_SIDE` (4096) px, and the chart's table as a DePlot linearised string of at most `MAX_TARGET_CHARS` (512) characters with at least one row; optional `image_id` groups records on the same chart (BYOD defaults it to the image file name) and optional `chart_type` labels the breakdown. Ids match `[A-Za-z0-9_.:-]` (1–64 characters) and are unique; a dataset needs 8..5,000 records; every record on the same chart lands in the same split. BYOD accepts one zip of images plus a `records.jsonl` / `records.json` in that shape.
- **Validation is structural, not semantic:** every image is opened and decoded and every target parsed, but nothing checks that a table is the chart's real data — a mislabelled chart is fine-tuned on without complaint, and SynthChartNet itself carries some label noise (for example a pie whose first category cell is a unit caption).
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path downloads one object from the Hub dataset repository `docling-project/SynthChartNet` at the immutable revision `b913ef98…` (`train-00000-of-00135.parquet`, 515,825,444 bytes) and refuses it unless its size and SHA-256 match the pins carried in `samples.py`; the chart images are written to the cache under their own content digest. SynthChartNet is published under CDLA-Permissive-2.0, which permits use and places no restriction on results such as trained weights; attribution to SynthChartNet (docling-project) is required when the data is shared.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/deplot` snapshot (~1133 MB in total) at revision `6e76d62430da…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'pyarrow==25.0.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'deplot-chart-pipeline',
    'repository_revision': '8d2ed5e529272cc006e3162701c98d78f57d59c6',
    'embedded_module': 'src/deplot_chart_pipeline/pipeline.py',
    'embedded_modules': ['src/deplot_chart_pipeline/pipeline.py', 'src/deplot_chart_pipeline/metrics.py', 'src/deplot_chart_pipeline/samples.py'],
    'module_sha256': '666339fa9527de993f1cf8997d6d249019da245b51648cbdd227a05d66579a03',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/deplot_chart_pipeline/` @ `8d2ed5e52927`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/deplot_chart_pipeline/pipeline.py`

In [ ]:
"""Chart-to-table extraction with the pinned ``google/deplot`` checkpoint (DePlot, a Pix2Struct model).

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the Pix2Struct architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed. The fixed instruction is rendered
as a text header on top of the chart (the Pix2Struct VQA input convention) with Pillow's bundled font,
so no font is fetched from the Hub at inference time.

The adaptation contract (`predict`, `evaluate`, `adapt`, `save_artifact`, `load_artifact`, `from_artifact`)
fine-tunes the decoder's last blocks on validated chart/table records with the linearised table as the
target, selects the epoch on validation cell accuracy and exports the trained tensors as a safetensors
adapter bound to the pinned base.
"""

from __future__ import annotations

import hashlib
import json
import math
import re
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image, ImageFont

MODEL_ID = "google/deplot"
MODEL_REVISION = "6e76d62430da16986be3426bae32301fb9115397"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "deplot"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = "ab90055611f42fee327d9ecf3c9cdac63e847bd19a0ac8ea86b0e8134fe0711b"
PARAMETER_COUNT = 282_285_696  # 18,879,744 of them train by default (2 decoder blocks + final norm)
DECODER_LAYERS = 12
DEFAULT_TRAINABLE_DECODER_LAYERS = 2
# Teacher-forcing ceiling for a target table (the SynthChartNet sample's longest target is 360 tokens).
MAX_TARGET_TOKENS = 512
# Evaluation bounds: a split larger than MAX_EVAL_RECORDS is refused (every chart is one generation of up
# to max_new_tokens); below MIN_SCORED_RECORDS the verdict says the sample is small.
MAX_EVAL_RECORDS = 500
MIN_SCORED_RECORDS = 20
ARTIFACT_FORMAT = "org.valcorza.deplot.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"

# The instruction the pinned README renders above the chart; DePlot was trained on this exact prompt
# and the pipeline does not expose any other.
INSTRUCTION = "Generate underlying data table of the figure below:"
# Generation ceilings. 512 is the max_new_tokens the pinned README's example passes; the ceiling
# leaves room for a long table.
MAX_NEW_TOKENS = 1024
DEFAULT_MAX_NEW_TOKENS = 512
DECODING = "greedy"
# Output conventions: DePlot linearises a table as rows separated by the literal token ``<0x0A>`` and
# cells separated by ``|``; the first row is ``TITLE | <chart title>`` when a title was read.
ROW_SEPARATOR = "<0x0A>"
CELL_SEPARATOR = "|"
# Input ceilings. The processor extracts at most MAX_PATCHES 16x16 patches (preprocessor_config.json)
# after scaling the image to fill that budget, so pixel count only guards memory during resizing.
MAX_PATCHES = 2048
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
# Relaxed numeric match tolerance for cell_accuracy (the ChartQA/DePlot "relaxed accuracy" convention).
RELATIVE_TOLERANCE = 0.05
_NUMBER_RE = re.compile(r"^[-+]?\$?\s*(\d[\d,]*\.?\d*|\.\d+)\s*%?$")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def header_font_bytes() -> bytes:
    """Pillow's bundled Aileron Regular (CC0) as TrueType bytes: the header font for the rendered question.

    The upstream image processor otherwise fetches ``ybelkada/fonts/Arial.TTF`` from the Hub at
    inference time — an unpinned, unlisted download of a proprietary font. The bundled subset covers
    the printable ASCII range, which is what a question is expected to use.
    """
    font = ImageFont.load_default(size=36)
    data = getattr(font, "font_bytes", None)
    if not data:
        raise RuntimeError("Pillow's bundled TrueType font is unavailable (FreeType support missing)")
    return bytes(data)


def parse_table(text: str) -> dict[str, Any]:
    """Split DePlot's linearised output into a title (or None) and a list of rows of stripped cells."""
    rows: list[list[str]] = []
    title: str | None = None
    for raw_row in text.split(ROW_SEPARATOR):
        cells = [cell.strip() for cell in raw_row.split(CELL_SEPARATOR)]
        if not any(cells):
            continue
        if title is None and not rows and len(cells) >= 2 and cells[0].upper() == "TITLE":
            title = CELL_SEPARATOR.join(cells[1:]).strip()
            continue
        rows.append(cells)
    return {
        "title": title,
        "rows": rows,
        "n_rows": len(rows),
        "n_columns": max((len(r) for r in rows), default=0),
    }


def _as_number(cell: str) -> float | None:
    match = _NUMBER_RE.match(cell.strip())
    if not match:
        return None
    try:
        return float(match.group(1).replace(",", ""))
    except ValueError:
        return None


def cells_match(predicted: str, expected: str, *, relative_tolerance: float = RELATIVE_TOLERANCE) -> bool:
    """Relaxed cell match: equal after case/whitespace normalisation, or numerically within the tolerance."""
    if " ".join(predicted.lower().split()) == " ".join(expected.lower().split()):
        return True
    p, e = _as_number(predicted), _as_number(expected)
    if p is None or e is None:
        return False
    return abs(p - e) <= relative_tolerance * abs(e) if e != 0 else abs(p) <= relative_tolerance


def cell_accuracy(
    predicted_rows: Sequence[Sequence[str]],
    expected_rows: Sequence[Sequence[str]],
    *,
    relative_tolerance: float = RELATIVE_TOLERANCE,
) -> dict[str, Any]:
    """Position-wise relaxed cell accuracy of a predicted table against the expected one.

    Every expected cell (row i, column j) counts once; it is matched only against the predicted cell at
    the same position (a missing row or column is a miss, an extra one is not penalised here but is
    reported through the shape fields). This is a sanity measure, not the paper's RMS metric.
    """
    if not expected_rows or not any(expected_rows):
        raise ValueError("expected_rows must contain at least one cell")
    total = matched = 0
    for i, expected in enumerate(expected_rows):
        predicted = predicted_rows[i] if i < len(predicted_rows) else []
        for j, cell in enumerate(expected):
            total += 1
            if j < len(predicted) and cells_match(predicted[j], cell, relative_tolerance=relative_tolerance):
                matched += 1
    return {
        "matched": matched,
        "total": total,
        "value": matched / total,
        "predicted_shape": [len(predicted_rows), max((len(r) for r in predicted_rows), default=0)],
        "expected_shape": [len(expected_rows), max((len(r) for r in expected_rows), default=0)],
        "relative_tolerance": relative_tolerance,
    }


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one chart image as PIL.Image.Image (any mode, converted to RGB): a bar, line or pie chart",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "instruction": INSTRUCTION,
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False), deterministic on a fixed device and dtype",
    "preprocessing": (
        "the fixed instruction is rendered as a black-on-white header (Pillow's bundled font) above the "
        "chart; the composite is scaled to fill at most MAX_PATCHES 16x16 patches (aspect ratio preserved), "
        "normalised per image, and flattened into patch tokens with row/column positions; the decoder "
        "generates the linearised table"
    ),
    "output": (
        f"linearised table text (rows separated by {ROW_SEPARATOR!r}, cells by {CELL_SEPARATOR!r}, optional "
        "leading TITLE row) plus its parsed rows; no score"
    ),
}


def _check_inputs(image: Any, max_new_tokens: Any) -> tuple[Image.Image, int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``extract_table`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, max_new_tokens


def validate_inputs(
    image: Image.Image,
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``extract_table`` would; a caller that wants the
    finding recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, checked_tokens = _check_inputs(image, max_new_tokens)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (extract_table takes one chart image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "instruction": INSTRUCTION,
        "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    expected_rows: Sequence[Sequence[str]] | None = None,
    *,
    expected_title: str | None = None,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``expected_rows`` (the table the chart really encodes, header row first) the report carries
    position-wise relaxed ``cell_accuracy`` and, when ``expected_title`` is given, a ``title_match``
    entry, verdict ``sample-sanity``; without ``expected_rows`` it is ``not-measurable`` and says what
    labelled data would make the task measurable.
    """
    table = result.get("table") or parse_table(str(result["text"]))
    base = {
        "task": "chart image -> linearised data table (plot-to-table)",
        "score_semantics": (
            "the table is generated text and carries no score, probability or correctness signal; a "
            "well-formed table is not evidence that its numbers are read from the chart. Greedy decoding "
            "makes the output reproducible on a fixed device and dtype, a reproducibility property, not a "
            "quality one"
        ),
        "sample_kind": sample_kind,
        "predicted_shape": [table["n_rows"], table["n_columns"]],
        "truncated": result.get("truncated"),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if expected_rows is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no expected data table was supplied for the evaluated chart",
            "needs": (
                "chart images paired with their underlying data tables from the deployment domain (chart "
                "types, styles, renderers) scored with relaxed cell accuracy or the DePlot paper's relative "
                "mapping similarity; no such labelled set ships with this repository"
            ),
        }
    accuracy = cell_accuracy(table["rows"], expected_rows)
    metrics: list[dict[str, Any]] = [
        {
            "id": "cell_accuracy",
            "value": accuracy["value"],
            "matched": accuracy["matched"],
            "total": accuracy["total"],
            "predicted_shape": accuracy["predicted_shape"],
            "expected_shape": accuracy["expected_shape"],
            "normalisation": (
                "position-wise; text cells compared case/whitespace-insensitively, numeric cells within "
                f"{RELATIVE_TOLERANCE:.0%} relative tolerance"
            ),
            "estimation": "one chart, no dispersion estimate",
        }
    ]
    if expected_title is not None:
        metrics.append(
            {
                "id": "title_match",
                "value": 1.0 if table["title"] and cells_match(table["title"], expected_title) else 0.0,
                "predicted": table["title"],
                "expected": expected_title,
                "estimation": "one chart, structural sanity only",
            }
        )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} sanity measure(s) on one tutorial chart whose data you rendered yourself; "
            "plumbing evidence, not a chart-to-table benchmark"
        ),
        "needs": (
            "a labelled chart/table set from the deployment domain (chart types, styles, renderers, "
            "languages) for any plot-to-table accuracy claim"
        ),
    }


@dataclass
class DePlotPipeline:
    """``_runner(image, max_new_tokens)`` returns ``{"text": str, "new_tokens": int}``; injectable so the
    offline tests run without the model."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)
    _font_bytes: bytes | None = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> DePlotPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        header_font_bytes()
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Pix2StructProcessor.from_pretrained(location, **common)
        if not getattr(processor.image_processor, "is_vqa", False):
            raise RuntimeError("snapshot image processor is not the VQA variant (is_vqa=False); refusing")
        # The model is loaded in float32 for CPU inference and training.
        model = Pix2StructForConditionalGeneration.from_pretrained(location, dtype=torch.float32, **common)
        return cls._from_model(model, processor, resolved_device, source)

    @classmethod
    def _from_model(cls, model: Any, processor: Any, device: str, source: str) -> DePlotPipeline:
        """Wrap a constructed model and VQA processor (every parameter frozen, eval mode) in a pipeline; the
        offline tests use it with a small randomly initialised Pix2Struct model."""
        import torch

        font_bytes = header_font_bytes()
        model = model.eval().to(device)
        for param in model.parameters():
            param.requires_grad_(False)

        def runner(image: Image.Image, max_new_tokens: int) -> dict[str, Any]:
            # The image processor is called directly: Pix2StructProcessor.__call__ drops the
            # font_bytes kwarg, and font_bytes is what replaces the default Hub font download
            # (see header_font_bytes). The VQA processor renders the instruction as the header.
            inputs = processor.image_processor(
                image, header_text=INSTRUCTION, return_tensors="pt", font_bytes=font_bytes
            ).to(device)
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1
                )
            # Encoder-decoder: the output holds only decoder tokens (decoder_start + table + eos).
            decoded = processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
            return {"text": decoded, "new_tokens": int(generated[0].shape[0]) - 1}

        return cls(
            runner, device, "float32", source, _model=model, _processor=processor, _font_bytes=font_bytes
        )

    def extract_table(
        self,
        image: Image.Image,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Translate one chart image into its linearised data table; ``table`` is the parsed form."""
        rgb, checked_tokens = _check_inputs(image, max_new_tokens)
        raw = self._runner(rgb, checked_tokens)
        if not isinstance(raw, dict) or "text" not in raw:
            raise RuntimeError("runner must return a dict with 'text'")
        text = str(raw["text"]).strip()
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "text": text,
            "table": parse_table(text),
            "instruction": INSTRUCTION,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation contract -----------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._processor

    def predict(
        self, records: Sequence[Mapping[str, Any]], *, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS
    ) -> list[dict[str, Any]]:
        """Extract the table of every validated record's chart; one `extract_table` result per record, in
        order, with the record's `id` and `chart_type` attached."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        out = []
        for record in checked:
            with Image.open(record["image"]) as image:
                image.load()
                result = self.extract_table(image, max_new_tokens=max_new_tokens)
            out.append({"id": record["id"], "chart_type": record["chart_type"], **result})
        return out

    def evaluate(
        self, records: Sequence[Mapping[str, Any]], *, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS
    ) -> dict[str, Any]:
        """Extract every record's table and score it against the record's target: mean relaxed position-wise
        cell accuracy, RNSS, exact-table match, the truncation rate and a per-chart-type breakdown (see
        `metrics.chart_metrics`)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import chart_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        results = self.predict(checked, max_new_tokens=max_new_tokens)
        metrics = chart_metrics([r["text"] for r in results], checked)
        metrics.update(
            {
                "truncated_rate": sum(bool(r["truncated"]) for r in results) / len(results),
                "max_new_tokens": max_new_tokens,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _decoder_layers(self) -> int:
        model, _ = self._require_model()
        return int(model.config.text_config.num_layers)

    def _trainable_names(self, trainable_decoder_layers: int) -> list[str]:
        """The last `trainable_decoder_layers` blocks of the text decoder plus the decoder's final layer norm.
        The untied output projection (`decoder.lm_head`, vocabulary x hidden) and every embedding stay frozen,
        as does the whole image encoder."""
        if (
            isinstance(trainable_decoder_layers, bool)
            or not isinstance(trainable_decoder_layers, int)
            or not 1 <= trainable_decoder_layers <= DECODER_LAYERS
        ):
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{DECODER_LAYERS}")
        model, _ = self._require_model()
        n_layers = self._decoder_layers()
        if trainable_decoder_layers > n_layers:
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{n_layers} for this model")
        first = n_layers - trainable_decoder_layers
        prefixes = tuple(f"decoder.layer.{k}." for k in range(first, n_layers))
        prefixes += ("decoder.final_layer_norm.",)
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def _encode_batch(self, records: Sequence[Mapping[str, Any]]) -> tuple[Any, Any]:
        """The frozen encoder's output and patch mask for a batch of (chart, rendered instruction) inputs.
        The header is part of the image, so every chart is its own encoder input; it is recomputed per step
        under `no_grad` instead of cached (2,048 x 768 floats per chart)."""
        import torch

        model, processor = self._require_model()
        device = next(model.parameters()).device
        images = []
        for record in records:
            with Image.open(record["image"]) as image:
                images.append(image.convert("RGB"))
        inputs = processor.image_processor(
            images, header_text=[INSTRUCTION] * len(images), return_tensors="pt", font_bytes=self._font_bytes
        ).to(device)
        with torch.no_grad():
            hidden = model.encoder(
                flattened_patches=inputs["flattened_patches"], attention_mask=inputs["attention_mask"]
            ).last_hidden_state
        return hidden, inputs["attention_mask"]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 3,
        lr: float = 1e-5,
        batch_size: int = 4,
        trainable_decoder_layers: int = DEFAULT_TRAINABLE_DECODER_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised fine-tuning on validated chart/table records.

        Only the last `trainable_decoder_layers` blocks of the text decoder and the decoder's final layer norm
        train (2 blocks by default); the image encoder, every embedding and the untied output projection stay
        frozen. Each record is one training sample: the fixed instruction is rendered above the chart exactly
        as `extract_table` renders it, the frozen encoder reads the composite, and the target is the tokenised
        linearised table (`target_text`, at most MAX_TARGET_TOKENS tokens) with its end-of-sequence token,
        decoded with teacher forcing and scored with the model's own cross-entropy (padding ignored); AdamW at
        a fixed learning rate with gradient clipping at 1.0, no scheduler. Epoch 0 records the frozen model's
        validation metrics; the epoch with the highest validation cell accuracy is kept (ties keep the
        earlier)."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        names = self._trainable_names(trainable_decoder_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        import torch

        torch.manual_seed(seed)
        model, processor = self._require_model()
        tokenizer = processor.tokenizer
        pad_id = int(tokenizer.pad_token_id)
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = next(model.parameters()).device

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {
                k: v
                for k, v in self.evaluate(val_checked).items()
                if k in ("cell_accuracy", "rnss", "exact_table_match", "n")
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_score = entry["val"]["cell_accuracy"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(train_checked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    chosen = [train_checked[j] for j in order[start : start + batch_size]]
                    hidden, mask = self._encode_batch(chosen)
                    targets = tokenizer(
                        [r["target_text"] for r in chosen],
                        padding=True,
                        truncation=True,
                        max_length=MAX_TARGET_TOKENS,
                        return_tensors="pt",
                    ).to(device)
                    labels = targets["input_ids"].masked_fill(targets["input_ids"] == pad_id, -100)
                    out = model(encoder_outputs=(hidden,), attention_mask=mask, labels=labels)
                    optimiser.zero_grad(set_to_none=True)
                    out.loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(float(out.loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
                history.append(entry)
                if progress:
                    progress(entry)
                current = entry["val"]["cell_accuracy"] if entry["val"] else math.inf
                if current > best_score or not entry["val"]:
                    best_score = current
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the base
            # exactly as it was, with every parameter frozen again.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_decoder_layers": trainable_decoder_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": (
                "highest validation cell accuracy" if val_checked else "final epoch (no validation split)"
            ),
            "lr": lr,
            "batch_size": batch_size,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ---------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {
            name: value.detach().cpu().contiguous()
            for name, value in model.state_dict().items()
            if name in names
        }
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {
                key: value
                for key, value in self.adapter.items()
                if key not in ("history", "trainable_names")
            },
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError("artifact format_version is not supported")
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file") != WEIGHT_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        layers = adapter.get("trainable_decoder_layers") if isinstance(adapter, Mapping) else None
        if isinstance(layers, bool) or not isinstance(layers, int) or not 1 <= layers <= DECODER_LAYERS:
            raise ValueError("artifact manifest does not record valid trainable decoder layers")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_decoder_layers"]))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("decoder."):
                raise ValueError(f"artifact tensor {key} is not an adaptable decoder tensor of the base")
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({key: value.to(state[key].dtype) for key, value in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> DePlotPipeline:
        pipeline = cls.from_pretrained(
            device=device,
            weights_dir=weights_dir,
            allow_download=allow_download,
        )
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 2/3:** `src/deplot_chart_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Chart-to-table scoring: relaxed position-wise cell accuracy, RNSS, exact-table match and a per-chart-type
breakdown, plus three non-neural baselines that never look at the chart image.

**Cell accuracy** (`pipeline.cell_accuracy`) matches every expected cell only against the predicted cell at
the same (row, column) position; text cells compare case- and whitespace-insensitively, numeric cells
within 5 % relative tolerance (the ChartQA "relaxed accuracy" convention). A missing row or column is a miss;
extra predicted rows or columns are not penalised (they show in the shapes). The ``TITLE`` row is not a cell.

**RNSS** — the relative number set similarity of Masry et al. (ChartQA, 2022, arXiv:2203.10244) as used for
plot-to-table evaluation by Liu et al. (DePlot, 2022, arXiv:2212.10505, §5.1): the numbers of the predicted
and the target table are matched one-to-one at minimal total cost, with the cost of a pair
``D(p, t) = min(1, |p - t| / |t|)``, and ``RNSS = 1 - cost / max(N, M)``. It ignores text cells and table
layout. Our reading of the unmatched case, stated because the formula leaves it implicit: a number left over
on the longer side costs the maximal distance 1, so emitting fewer or more numbers than the target lowers the
score; two tables without numbers score 1. The paper's RMS (relative mapping similarity, which also matches
row and column headers) is not implemented.

**Exact-table match** is the fraction of charts whose parsed rows equal the target's after lower-casing and
whitespace collapsing.

The baselines see only the training split and the test chart's type — never its image or its target. The
**empty baseline** emits no table (the floor: cell accuracy 0). The **header-only baseline** emits, per chart
type, the most frequent header row of the training targets (a header being a first row with an empty corner
cell) and no data rows. The **medoid baseline** emits, per chart type, the training target with the highest
mean cell accuracy against the other training targets of that type — the single most "typical" table. A
system that does not beat them has not shown that it reads the chart.
"""

from __future__ import annotations

import statistics
from collections import Counter, defaultdict
from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import CELL_SEPARATOR, ROW_SEPARATOR, _as_number, cell_accuracy, parse_table` removed — names are kernel globals defined by the carried modules

METRIC_DEFINITIONS: dict[str, str] = {
    "cell_accuracy": (
        "mean over charts of the relaxed position-wise cell accuracy (text case/whitespace-insensitive, "
        "numbers within 5 % relative tolerance; missing cells are misses, extra cells are not penalised)"
    ),
    "rnss": (
        "mean over charts of the relative number set similarity (ChartQA / DePlot): minimal-cost one-to-one "
        "matching of the tables' numbers with cost min(1, |p - t| / |t|), unmatched numbers at cost 1"
    ),
    "exact_table_match": "fraction of charts whose normalised parsed rows equal the target's",
    "by_chart_type": "the same means per chart type",
}


def _normalise_rows(rows: Sequence[Sequence[str]]) -> tuple[tuple[str, ...], ...]:
    return tuple(tuple(" ".join(cell.lower().split()) for cell in row) for row in rows)


def _numbers(rows: Sequence[Sequence[str]]) -> list[float]:
    return [value for row in rows for cell in row if (value := _as_number(cell)) is not None]


def _min_cost_assignment(cost: Sequence[Sequence[float]]) -> float:
    """Total cost of the minimal-cost perfect matching of a square cost matrix (Hungarian algorithm,
    O(n^3), potentials form)."""
    n = len(cost)
    if n == 0:
        return 0.0
    inf = float("inf")
    u, v = [0.0] * (n + 1), [0.0] * (n + 1)
    match, way = [0] * (n + 1), [0] * (n + 1)
    for i in range(1, n + 1):
        match[0], j0 = i, 0
        minv, used = [inf] * (n + 1), [False] * (n + 1)
        while True:
            used[j0] = True
            i0, delta, j1 = match[j0], inf, 0
            for j in range(1, n + 1):
                if not used[j]:
                    current = cost[i0 - 1][j - 1] - u[i0] - v[j]
                    if current < minv[j]:
                        minv[j], way[j] = current, j0
                    if minv[j] < delta:
                        delta, j1 = minv[j], j
            for j in range(n + 1):
                if used[j]:
                    u[match[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if match[j0] == 0:
                break
        while True:
            j1 = way[j0]
            match[j0] = match[j1]
            j0 = j1
            if j0 == 0:
                break
    return sum(cost[match[j] - 1][j - 1] for j in range(1, n + 1))


def rnss(predicted_rows: Sequence[Sequence[str]], expected_rows: Sequence[Sequence[str]]) -> float:
    """Relative number set similarity of two tables (see the module docstring for the definition)."""
    predicted, expected = _numbers(predicted_rows), _numbers(expected_rows)
    size = max(len(predicted), len(expected))
    if size == 0:
        return 1.0

    def distance(p: float, t: float) -> float:
        if t == 0:
            return 0.0 if p == 0 else 1.0
        return min(1.0, abs(p - t) / abs(t))

    cost = [
        [
            distance(predicted[i], expected[j]) if i < len(predicted) and j < len(expected) else 1.0
            for j in range(size)
        ]
        for i in range(size)
    ]
    return 1.0 - _min_cost_assignment(cost) / size


def score_table(prediction: str, expected: str | Mapping[str, Any]) -> dict[str, Any]:
    """Cell accuracy, RNSS and exact match of one predicted linearised table against its target."""
    target = expected if isinstance(expected, Mapping) else parse_table(str(expected))
    predicted = parse_table(str(prediction))
    accuracy = cell_accuracy(predicted["rows"], target["rows"])
    return {
        "cell_accuracy": accuracy["value"],
        "rnss": rnss(predicted["rows"], target["rows"]),
        "exact_table_match": _normalise_rows(predicted["rows"]) == _normalise_rows(target["rows"]),
        "predicted_shape": accuracy["predicted_shape"],
        "expected_shape": accuracy["expected_shape"],
    }


def chart_metrics(predictions: Sequence[str], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Aligned predictions scored against the records' targets: means overall and per chart type, plus one
    row per chart."""
    if len(predictions) != len(records):
        raise ValueError(f"{len(predictions)} predictions for {len(records)} records")
    if not records:
        raise ValueError("records must not be empty")
    rows = []
    for prediction, record in zip(predictions, records, strict=True):
        scored = score_table(prediction, record.get("table") or str(record["target_text"]))
        rows.append(
            {
                "id": record["id"],
                "chart_type": str(record.get("chart_type", "other")),
                "prediction": str(prediction),
                "target_text": str(record["target_text"]),
                **scored,
            }
        )
    groups: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in rows:
        groups[row["chart_type"]].append(row)

    def means(part: Sequence[Mapping[str, Any]]) -> dict[str, float]:
        return {
            "cell_accuracy": round(statistics.fmean(r["cell_accuracy"] for r in part), 4),
            "rnss": round(statistics.fmean(r["rnss"] for r in part), 4),
            "exact_table_match": round(statistics.fmean(float(r["exact_table_match"]) for r in part), 4),
        }

    return {
        "n": len(rows),
        **means(rows),
        "by_chart_type": {name: {"n": len(part), **means(part)} for name, part in sorted(groups.items())},
        "rows": rows,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def _format(rows: Sequence[Sequence[str]]) -> str:
    """DePlot's linearisation with an empty title (the same as `samples.format_table`)."""
    lines = [f"TITLE {CELL_SEPARATOR} ", *(f" {CELL_SEPARATOR} ".join(row) for row in rows)]
    return f" {ROW_SEPARATOR} ".join(lines)


def _by_type(records: Sequence[Mapping[str, Any]]) -> dict[str, list[Mapping[str, Any]]]:
    groups: dict[str, list[Mapping[str, Any]]] = defaultdict(list)
    for record in records:
        groups[str(record.get("chart_type", "other"))].append(record)
    return groups


def _score(name: str, note: str, predictions: list[str], test: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    return {**chart_metrics(predictions, test), "baseline": name, "note": note}


def empty_baseline(test: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Emit no table for every test chart."""
    if not test:
        raise ValueError("test must be non-empty")
    return _score("empty", "no table at all (the floor)", [""] * len(test), test)


def header_only_baseline(
    train: Sequence[Mapping[str, Any]], test: Sequence[Mapping[str, Any]]
) -> dict[str, Any]:
    """Emit, per chart type, the most frequent training header row and no data rows."""
    if not train or not test:
        raise ValueError("train and test must be non-empty")
    headers: dict[str, str] = {}
    for chart_type, part in _by_type(train).items():
        counts = Counter(
            tuple(rows[0])
            for rows in (parse_table(str(r["target_text"]))["rows"] for r in part)
            if rows and rows[0] and rows[0][0] == ""
        )
        headers[chart_type] = (
            _format([list(min(counts, key=lambda k: (-counts[k], k)))]) if counts else _format([])
        )
    predictions = [headers.get(str(r.get("chart_type", "other")), _format([])) for r in test]
    note = "the most frequent training header row of the chart's type, no data rows"
    return {**_score("header-only", note, predictions, test), "headers": headers}


def medoid_baseline(train: Sequence[Mapping[str, Any]], test: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Emit, per chart type, the training target with the highest mean cell accuracy against the other
    training targets of that type (the overall medoid for a type absent from training)."""
    if not train or not test:
        raise ValueError("train and test must be non-empty")

    def medoid(part: Sequence[Mapping[str, Any]]) -> str:
        tables = [parse_table(str(r["target_text"]))["rows"] for r in part]
        if len(tables) == 1:
            return str(part[0]["target_text"])
        best, best_score = 0, -1.0
        for i, candidate in enumerate(tables):
            score = statistics.fmean(
                cell_accuracy(candidate, other)["value"] for j, other in enumerate(tables) if j != i and other
            )
            if score > best_score:
                best, best_score = i, score
        return str(part[best]["target_text"])

    tables = {chart_type: medoid(part) for chart_type, part in sorted(_by_type(train).items())}
    fallback = medoid(train)
    predictions = [tables.get(str(r.get("chart_type", "other")), fallback) for r in test]
    note = "the training medoid table of the chart's type (highest mean cell accuracy against its type)"
    return {**_score("medoid", note, predictions, test), "tables": tables}

**Module 3/3:** `src/deplot_chart_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Chart-to-table dataset contract for fine-tuning: the pinned SynthChartNet sample, the OTSL-to-DePlot target
converter, validation, seeded chart-type-stratified splitting, BYOD loaders and JSONL export.

The default dataset is **real** chart/table pairs: charts from `docling-project/SynthChartNet` (bar, pie,
stacked-bar and line charts rendered with Matplotlib, Seaborn and Pyecharts, each paired with the data table
it was drawn from, published under **CDLA-Permissive-2.0**). SynthChartNet is not part of DePlot's
fine-tuning mixture (synthetic, ChartQA and PlotQA plots), so this is adaptation to a new chart family, not a
re-run of the checkpoint's own data. The sample is one pinned parquet shard (`train-00000-of-00135.parquet`,
515,825,444 bytes) downloaded whole at the pinned dataset revision and refused unless its SHA-256 matches
the pin before `pyarrow` reads a byte of it. A seeded, chart-type-stratified subset of charts is drawn from
it and cut into training, validation and test charts; every chart image is written to the cache under its
own content digest, and a chart whose image digest was already drawn is skipped, so no image is shared
between splits.

**Target format — the model's own convention, not an assumption.** SynthChartNet stores each table as OTSL
(``<fcel>`` a cell, ``<ecel>`` an empty cell, ``<nl>`` a row end, after ``<loc_…>`` box tokens and a
``<{bar|pie|stacked_bar|line}_chart>`` type tag). The pinned DePlot checkpoint, run frozen on bar, pie,
stacked-bar and line charts of this shard, always emits ``TITLE | <title>`` first (the title empty: these
charts carry none), then one row per **axis category / x point** with the series as columns, cells joined by
`` | `` and rows by `` <0x0A> ``; it writes a header row (empty corner, series names) when the chart has a
legend or a series name and none for a single unnamed series (pies). OTSL orients bar, pie and stacked-bar
tables the other way — the axis categories run along the first row and each further row is one series — while
line tables already hold one row per x point under a ``<ecel> | series`` header. The converter
(`otsl_to_rows`, `target_from_otsl`) therefore applies **one rule**:

* bar, pie, stacked_bar: transpose the OTSL grid; line: keep it;
* ``<ecel>`` becomes an empty cell; nothing else is dropped, merged or renamed;
* the target is ``TITLE | `` followed by the oriented rows, cells joined by `` | `` and rows by `` <0x0A> ``.

A two-row single-series bar or pie table thus becomes ``category | value`` rows with no header (it has no
series name), and a table with an ``<ecel>`` corner keeps a header row ``  | series …``. The literal
``<0x0A>`` in a target tokenises to the same byte token the model emits.

A record is ``{id, image_id, image, target_text, chart_type}`` — the path of the chart image, the linearised
target table, and the chart type (`bar`, `pie`, `stacked_bar`, `line`; BYOD records default to `other`).

The shard's SHA-256 and row count are recorded by `tools/pin_corpus.py`; until they are recorded,
`fetch_corpus` refuses to read the shard rather than read an unpinned file.
"""

from __future__ import annotations

import hashlib
import io
import json
import random
import re
from collections import Counter
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import CELL_SEPARATOR, MODEL_ID, ROW_SEPARATOR, parse_table, validate_image` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "SynthChartNet"
CORPUS_REPO = "docling-project/SynthChartNet"
CORPUS_REVISION = "b913ef98a3d45f6136465963ddc71a7a6b0e1728"
CORPUS_RELEASE = (
    "SynthChartNet training shard 0 of 135 (14,568 charts with their OTSL data tables) on the Hugging Face "
    "Hub, dataset revision b913ef98"
)
CORPUS_LICENSE = (
    "CDLA-Permissive-2.0 (declared in the Hub dataset card): use, modification and sharing permitted, "
    "no restriction on results such as trained weights; attribution to SynthChartNet (docling-project) "
    "required"
)
CORPUS_COLUMNS = ("images", "texts")
# The shard's pins. `sha256` and `rows` are written by tools/pin_corpus.py from a verified file; while
# `sha256` is None the reader refuses to run.
CORPUS_FILE: dict[str, Any] = {
    "path": "train-00000-of-00135.parquet",
    "bytes": 515_825_444,
    "sha256": "1d6cf578fdd953fa0dd7c1d169926eb3418daa2770cc3a301b1a64bec0b209e7",
    "rows": 14568,
}
DEFAULT_CACHE_DIR = Path("weights") / "synthchartnet"
SAMPLE_SEED = 42
# Chart counts per split; each split is stratified by chart type in proportion to the shard's type mix
# (largest remainder), so a rare type can receive no chart in a small split.
SAMPLE_CHARTS = {"train": 360, "validation": 80, "test": 160}
CHART_TYPES = ("bar", "pie", "stacked_bar", "line")
# OTSL orients these types with the axis categories along the first row; DePlot emits one row per category.
TRANSPOSED_TYPES = frozenset({"bar", "pie", "stacked_bar"})
MIN_RECORDS = 8
MAX_RECORDS = 5_000
# Target ceiling: the linearised table is also the decoder's teacher-forcing target, so it is bounded to
# keep it well inside MAX_TARGET_TOKENS of the pipeline (the shard's longest target under it is 360 tokens).
MAX_TARGET_CHARS = 512
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
_TYPE_RE = re.compile(r"<(bar|pie|stacked_bar|line)_chart>")
_CELL_RE = re.compile(r"<(fcel|ecel)>([^<]*)")
_TITLE_ROW = f"TITLE {CELL_SEPARATOR} "
_ROW_JOIN = f" {ROW_SEPARATOR} "
_CELL_JOIN = f" {CELL_SEPARATOR} "


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


# ---- the OTSL -> DePlot target converter ------------------------------------------------------


def parse_otsl(otsl: str) -> dict[str, Any]:
    """The chart type and the OTSL grid (rows of cell strings, ``<ecel>`` as ``""``) of one SynthChartNet
    table; raises ValueError on a missing type tag, an unknown tag, an empty or a ragged grid."""
    if not isinstance(otsl, str):
        raise ValueError("OTSL table must be a string")
    match = _TYPE_RE.search(otsl)
    if not match:
        raise ValueError("OTSL table has no <{bar|pie|stacked_bar|line}_chart> tag")
    body = otsl[match.end() :].replace("</chart>", "")
    grid: list[list[str]] = []
    for raw_row in body.split("<nl>"):
        if not raw_row.strip():
            continue
        cells = _CELL_RE.findall(raw_row)
        if "".join(f"<{kind}>{text}" for kind, text in cells) != raw_row:
            raise ValueError(f"OTSL row holds a tag other than <fcel>/<ecel>: {raw_row[:60]!r}")
        grid.append([" ".join(text.split()) if kind == "fcel" else "" for kind, text in cells])
    if not grid:
        raise ValueError("OTSL table has no rows")
    if len({len(row) for row in grid}) != 1:
        raise ValueError("OTSL table is ragged (rows of different widths)")
    if any(CELL_SEPARATOR in cell or ROW_SEPARATOR in cell for row in grid for cell in row):
        raise ValueError("OTSL cell contains a DePlot separator")
    return {"chart_type": match.group(1), "rows": grid}


def otsl_to_rows(otsl: str) -> list[list[str]]:
    """The OTSL grid oriented as DePlot emits it: transposed for bar, pie and stacked-bar charts (one row per
    axis category), unchanged for line charts (already one row per x point)."""
    parsed = parse_otsl(otsl)
    grid = parsed["rows"]
    if parsed["chart_type"] in TRANSPOSED_TYPES:
        grid = [list(column) for column in zip(*grid, strict=True)]
    return grid


def format_table(rows: Sequence[Sequence[str]], title: str = "") -> str:
    """DePlot's linearisation: ``TITLE | <title>``, then the rows; cells joined by `` | ``, rows by
    `` <0x0A> ``."""
    return _ROW_JOIN.join([_TITLE_ROW + title, *(_CELL_JOIN.join(row) for row in rows)])


def target_from_otsl(otsl: str) -> str:
    """The training/evaluation target of one SynthChartNet table (see the module docstring for the rule)."""
    return format_table(otsl_to_rows(otsl))


# ---- the pinned shard ----


def corpus_pinned() -> bool:
    """Whether the shard's SHA-256 has been recorded (see tools/pin_corpus.py)."""
    return isinstance(CORPUS_FILE.get("sha256"), str) and len(CORPUS_FILE["sha256"]) == 64


def _hub_download(cache: Path) -> Path:
    from huggingface_hub import hf_hub_download

    return Path(
        hf_hub_download(
            CORPUS_REPO,
            CORPUS_FILE["path"],
            repo_type="dataset",
            revision=CORPUS_REVISION,
            local_dir=str(cache),
        )
    )


def fetch_corpus(
    *, cache_dir: str | Path | None = None, downloader: Callable[[Path], Path] | None = None
) -> Path:
    """Return the path of the pinned shard, downloading it at the pinned revision when the cached copy is
    absent or drifted; refused on any size or SHA-256 mismatch, and outright while no pin is recorded."""
    if not corpus_pinned():
        raise RuntimeError(
            f"{CORPUS_REPO}@{CORPUS_REVISION[:8]} {CORPUS_FILE['path']}: no SHA-256 pin is recorded; run "
            "tools/pin_corpus.py with Hub access to record it before the sample can be read"
        )
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / CORPUS_FILE["path"]

    def ok(path: Path) -> bool:
        return (
            path.is_file()
            and path.stat().st_size == CORPUS_FILE["bytes"]
            and _sha256_file(path) == CORPUS_FILE["sha256"]
        )

    if ok(local):
        return local
    fetched = (downloader or _hub_download)(cache)
    if not ok(fetched):
        size = fetched.stat().st_size if fetched.is_file() else None
        raise ValueError(
            f"{CORPUS_FILE['path']}: fetched {size} bytes, pinned {CORPUS_FILE['bytes']} / "
            f"{CORPUS_FILE['sha256'][:16]}…; refusing to read it"
        )
    return fetched


def read_corpus(path: str | Path) -> list[dict[str, Any]]:
    """The shard's tables as ``{row, chart_type, otsl}`` (text column only; images are read per chosen row by
    `build_sample_dataset`). A row whose OTSL does not parse keeps ``chart_type`` None and is never drawn."""
    import pyarrow.parquet as pq

    texts = pq.read_table(str(path), columns=["texts"]).column("texts").to_pylist()
    out = []
    for index, entry in enumerate(texts):
        if not entry or not isinstance(entry[0], Mapping) or "assistant" not in entry[0]:
            raise ValueError(f"row {index}: no assistant table")
        otsl = str(entry[0]["assistant"])
        try:
            chart_type = parse_otsl(otsl)["chart_type"]
        except ValueError:
            chart_type = None
        out.append({"row": index, "chart_type": chart_type, "otsl": otsl})
    if CORPUS_FILE.get("rows") is not None and len(out) != CORPUS_FILE["rows"]:
        raise ValueError(f"shard has {len(out)} rows, pinned {CORPUS_FILE['rows']}")
    return out


def _image_column(shard_path: str | Path) -> Any:
    import pyarrow.parquet as pq

    return pq.read_table(str(shard_path), columns=["images"]).column("images")


def _image_bytes(column: Any, row: int) -> bytes:
    images = column[row].as_py()
    data = images[0].get("bytes") if images and isinstance(images[0], Mapping) else None
    if not data:
        raise ValueError(f"row {row}: no image bytes")
    return bytes(data)


def _image_suffix(data: bytes) -> str:
    with Image.open(io.BytesIO(data)) as image:
        fmt = (image.format or "PNG").lower()
    return {"jpeg": ".jpg", "png": ".png", "gif": ".gif", "webp": ".webp"}.get(fmt, ".png")


def stratified_quotas(type_counts: Mapping[str, int], size: int) -> dict[str, int]:
    """Split `size` over the chart types in proportion to `type_counts` (largest remainder, ties by the
    order of CHART_TYPES then name)."""
    total = sum(type_counts.values())
    if total <= 0:
        raise ValueError("no charts to stratify")
    names = sorted(
        type_counts, key=lambda k: (CHART_TYPES.index(k) if k in CHART_TYPES else len(CHART_TYPES), k)
    )
    exact = {k: size * type_counts[k] / total for k in names}
    quotas = {k: int(exact[k]) for k in names}
    for k in sorted(names, key=lambda k: -(exact[k] - quotas[k]))[: size - sum(quotas.values())]:
        quotas[k] += 1
    return quotas


def build_sample_dataset(
    rows: Sequence[Mapping[str, Any]],
    shard_path: str | Path,
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
    image_dir: str | Path | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Shuffle each chart type's rows with `seed`, give every split its type quota (`stratified_quotas` of
    the shard's type mix) in the order test, validation, train, and write the chosen chart images to
    `image_dir` under their digest. Charts `validate_dataset` would reject (a target over
    `MAX_TARGET_CHARS`, an image side outside the ceilings) and charts whose image digest was already drawn
    are left out before they are counted, so every split validates and no image is shared."""
    sizes = dict(sizes or SAMPLE_CHARTS)
    out_dir = Path(image_dir) if image_dir is not None else DEFAULT_CACHE_DIR / "images"
    out_dir.mkdir(parents=True, exist_ok=True)
    pools: dict[str, list[tuple[int, str]]] = {}
    for row in rows:
        if row.get("chart_type") is None:
            continue
        target = target_from_otsl(str(row["otsl"]))
        if len(target) <= MAX_TARGET_CHARS:
            pools.setdefault(str(row["chart_type"]), []).append((int(row["row"]), target))
    type_counts = {name: len(pool) for name, pool in pools.items()}
    quotas = {name: stratified_quotas(type_counts, sizes[name]) for name in ("test", "validation", "train")}
    for name in sorted(pools):
        random.Random(f"{seed}:{name}").shuffle(pools[name])
    out: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    seen: set[str] = set()
    column = _image_column(shard_path)
    for chart_type in sorted(pools):
        candidates = iter(pools[chart_type])
        for split in ("test", "validation", "train"):
            want, taken = quotas[split].get(chart_type, 0), 0
            while taken < want:
                candidate = next(candidates, None)
                if candidate is None:
                    break
                row, target = candidate
                data = _image_bytes(column, row)
                image_id = _sha256_bytes(data)[:16]
                if image_id in seen:
                    continue
                path = out_dir / f"{image_id}{_image_suffix(data)}"
                if not path.is_file() or _sha256_bytes(path.read_bytes())[:16] != image_id:
                    path.write_bytes(data)
                record = {
                    "id": f"{split}-{len(out[split]):04d}",
                    "image_id": image_id,
                    "image": str(path),
                    "target_text": target,
                    "chart_type": chart_type,
                    "row": row,
                }
                try:
                    _check_record(record, 0, base_dir=None)
                except ValueError:
                    continue  # outside the ceilings `validate_dataset` and `extract_table` apply
                seen.add(image_id)
                out[split].append(record)
                taken += 1
    short = {name: (len(out[name]), sizes[name]) for name in out if len(out[name]) < sizes[name]}
    if short:
        raise ValueError(f"the shard's charts do not fill the split targets: {short}")
    return {"train": out["train"], "validation": out["validation"], "test": out["test"]}


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    downloader: Callable[[Path], Path] | None = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned shard."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    shard = fetch_corpus(cache_dir=cache, downloader=downloader)
    return build_sample_dataset(read_corpus(shard), shard, seed=seed, sizes=sizes, image_dir=cache / "images")


# ---- dataset contract ----


def _check_record(record: Any, index: int, *, base_dir: Path | None) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/image/target_text")
    for key in ("id", "image", "target_text"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    image_ref = record["image"]
    if not isinstance(image_ref, (str, Path)) or not str(image_ref).strip():
        raise ValueError(f"{label}: image must be a file path")
    path = Path(image_ref)
    if not path.is_absolute() and base_dir is not None:
        path = base_dir / path
    if not path.is_file():
        raise ValueError(f"{label}: image file not found: {path}")
    try:
        with Image.open(path) as handle:
            handle.load()
            validate_image(handle)
            width, height = handle.size
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label}: {exc}") from exc
    except OSError as exc:
        raise ValueError(f"{label}: image cannot be decoded: {exc}") from exc
    target = record["target_text"]
    if not isinstance(target, str) or not target.strip():
        raise ValueError(f"{label}: target_text must be a non-empty string")
    target = target.strip()
    if len(target) > MAX_TARGET_CHARS:
        raise ValueError(f"{label}: target_text exceeds MAX_TARGET_CHARS={MAX_TARGET_CHARS}")
    table = parse_table(target)
    if not table["rows"]:
        raise ValueError(f"{label}: target_text must hold at least one table row")
    return {
        "id": rid,
        "image_id": str(record.get("image_id", path.name)),
        "image": str(path),
        "image_size": [width, height],
        "target_text": target,
        "table": table,
        "chart_type": str(record.get("chart_type", "other")),
    }


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    base_dir: str | Path | None = None,
) -> dict[str, Any]:
    """Structural validation of a chart/table dataset (every image opened and decoded under the ceilings
    `extract_table` applies, every target parsed into at least one row); raises ValueError before any model
    import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, image, target_text} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    base = Path(base_dir) if base_dir is not None else None
    checked = []
    ids: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index, base_dir=base)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        checked.append(item)
    shapes = [(r["table"]["n_rows"], r["table"]["n_columns"]) for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_images": len({r["image_id"] for r in checked}),
        "chart_types": dict(Counter(r["chart_type"] for r in checked)),
        "table_rows": {"min": min(s[0] for s in shapes), "max": max(s[0] for s in shapes)},
        "table_columns": {"min": min(s[1] for s in shapes), "max": max(s[1] for s in shapes)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [
        [r["id"], r.get("image_id", ""), str(r["target_text"]).strip(), r.get("chart_type", "")]
        for r in records
    ]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no chart image appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record.get("image_id", record["id"]))
            if key in seen and seen[key] != name:
                raise ValueError(f"chart {key!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
    base_dir: str | Path | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded split of a BYOD dataset into train/validation/test **by chart image**: records on the same
    `image_id` land in the same split, so a test chart is never seen in training."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records, base_dir=base_dir)["records"]
    groups: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        groups.setdefault(record["image_id"], []).append(record)
    order = list(groups.values())
    random.Random(seed).shuffle(order)
    n_test = max(1, round(len(checked) * test_fraction))
    n_val = round(len(checked) * val_fraction)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for group in order:
        if len(splits["test"]) < n_test:
            splits["test"].extend(group)
        elif len(splits["validation"]) < n_val:
            splits["validation"].extend(group)
        else:
            splits["train"].extend(group)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read records from a JSON array or a JSONL file of ``{id, image, target_text}`` objects; `image` paths
    are resolved relative to the file's directory by `validate_dataset(..., base_dir=...)`."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    raise ValueError("BYOD datasets must be .json or .jsonl")


def write_dataset_jsonl(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """One record per line in the shape `load_byod_dataset` reads back (image paths as given)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    keys = ("id", "image_id", "image", "target_text", "chart_type")
    with open(out, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps({k: record[k] for k in keys if k in record}, ensure_ascii=False) + "\n")
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `6e76d62430da…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `DePlotPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "deplot",
  "modelId": "google/deplot",
  "revision": "6e76d62430da16986be3426bae32301fb9115397",
  "files": [
    {
      "path": "README.md",
      "bytes": 4317,
      "sha256": "419a84efe72647da6e56637a4b92394b3b3df6313d8b233c5ddc9942aad80102"
    },
    {
      "path": "config.json",
      "bytes": 4883,
      "sha256": "f1dc8bcbda2ac4f8715c2d5003d3afb591ac0af97782fdf95a65cf26f805c5bb"
    },
    {
      "path": "model.safetensors",
      "bytes": 1129177976,
      "sha256": "ab90055611f42fee327d9ecf3c9cdac63e847bd19a0ac8ea86b0e8134fe0711b"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 249,
      "sha256": "c84e4eebc84171d6069533d9f0147ec7b4afd02ab78697cb5c30f9419ef7dc45"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2201,
      "sha256": "5c87151ef0f72a99d1f766a4c418bd2a1f90aaa30a8e22fe5eca9641daebb64f"
    },
    {
      "path": "spiece.model",
      "bytes": 851388,
      "sha256": "7fd650335add59bed55a432186ca0437a09e185c2d241faab468a538fe6bcf94"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3265159,
      "sha256": "0af109b23840545ef2c286073f4373959badba1faa73c8557881d5126f6287c9"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 2623,
      "sha256": "73949c3b853a39cd4661303b0e5cb7b836f7d783be6e898f8b7ae89e203cbe73"
    }
  ],
  "totalBytes": 1133308796
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = DePlotPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. SynthChartNet charts, tables and split

`fetch_corpus` returns the pinned shard from the cache under `weights/synthchartnet/` or downloads it at the pinned dataset revision, and refuses it unless its byte size and SHA-256 equal the pins in the carried module (it also refuses to run at all while no SHA-256 pin is recorded). `read_corpus` reads the table column with `pyarrow` and parses every OTSL table. `build_sample_dataset` converts each table to its target with `target_from_otsl` — bar, pie and stacked-bar tables transposed to one row per category, line tables kept, `<ecel>` as an empty cell, `TITLE | ` first — leaves out the few charts whose target exceeds `MAX_TARGET_CHARS`, shuffles each chart type with `SPLIT_SEED`, and fills the test, validation and training splits to `SAMPLE_CHARTS` in proportion to the shard's type mix (so line charts, 0.5 % of the shard, get one or two charts per split, or none). A chart whose image digest was already drawn is skipped, so no image is shared between splits. `validate_dataset` then opens every image and parses every target, `check_split_disjoint` asserts no image is shared, and the training split is written to `outputs/deplot_chart_train.jsonl` in the shape BYOD expects.

Look for: the shard's row count and type mix, one raw OTSL table beside its converted target, the chart counts per split and per type, the table shapes, three digests, and four refusal probes — a duplicate id, a missing image file, a target with no table row and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import collections
import io
import json
import zipfile

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_dir = Path('work') / 'byod'
    byod_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            name = Path(member.filename).name
            if member.is_dir() or not name or name.startswith('.'):
                continue
            (byod_dir / name).write_bytes(archive.read(member))
    records_file = next(p for p in (byod_dir / 'records.jsonl', byod_dir / 'records.json') if p.is_file())
    records = load_byod_dataset(records_file)
    splits = split_dataset(records, seed=SPLIT_SEED, base_dir=byod_dir)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    shard_path = fetch_corpus(cache_dir='weights/synthchartnet')
    corpus_rows = read_corpus(shard_path)
    raw_rows = {'charts': len(corpus_rows), 'chart_types': dict(collections.Counter(r['chart_type'] for r in corpus_rows))}
    example_row = next(r for r in corpus_rows if r['chart_type'] == 'stacked_bar')
    print({'otsl': example_row['otsl'][:220], 'target': target_from_otsl(example_row['otsl'])[:220]})
    splits = build_sample_dataset(corpus_rows, shard_path, seed=SPLIT_SEED, image_dir='weights/synthchartnet/images')
    data_source = f'{CORPUS_NAME} — {CORPUS_RELEASE}'
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
chart_types = {name: manifest['chart_types'] for name, manifest in dataset_manifests.items()}
write_dataset_jsonl(splits['train'], 'outputs/deplot_chart_train.jsonl')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'shard_sha256': str(CORPUS_FILE['sha256'])[:16] + '...'})
for name, manifest in dataset_manifests.items():
    print({name: {'charts': manifest['n_records'], 'images': manifest['unique_images'], 'chart_types': manifest['chart_types'], 'table_rows': manifest['table_rows'], 'table_columns': manifest['table_columns'], 'digest': manifest['digest'][:16] + '...'}})
example = splits['train'][0]
print({'example': {'id': example['id'], 'image': Path(example['image']).name, 'size': example['image_size'], 'chart_type': example['chart_type'], 'target_text': example['target_text'][:200]}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in splits['train'][:8]],
    'missing image file': [{**splits['train'][0], 'image': 'work/does-not-exist.png'}, *splits['train'][1:8]],
    'target without a table row': [{**splits['train'][0], 'target_text': 'TITLE | nothing else'}, *splits['train'][1:8]],
    'too small': splits['train'][:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Extract a table through the inference contract

The inference contract is exercised as the inference-only tutorial exercised it: a bar chart drawn in code at 800×520 from the five-row table `EXPECTED_TABLE` — the title `Quarterly revenue 2025 (USD millions)`, a y axis from 0 to 200 with gridlines, four bars for Q1–Q4 with their values printed above them. It is a different image family from the SynthChartNet charts, and the adapted model will extract it again in Section 9. `validate_inputs` applies exactly the checks `extract_table` applies (image sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE`, `max_new_tokens` in `[1, MAX_NEW_TOKENS]`) and returns an input manifest; a zero budget is validated too and its rejection recorded as a finding. `extract_table` returns `text` (the linearised table exactly as decoded), the parsed `table`, `new_tokens`, a `truncated` flag that is true when the budget was exhausted, and the model identity. **No score exists.** As recorded in the model card, the repository's CPU smoke on this chart generated 56 tokens: the title and all four quarters and values exactly, and the header `Quarter | Quarterly revenue` where the reference says `Revenue`. `evaluation_report` scores it against the table you drew yourself — verdict `sample-sanity`, plumbing evidence, not a metric. The image digest depends on the Pillow build's bundled font rendering.

In [ ]:
import csv
import hashlib
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont

TABLE_MAX_TOKENS = 512  # @param {type:"integer"}
CHART_TITLE = 'Quarterly revenue 2025 (USD millions)'
EXPECTED_TABLE = [['Quarter', 'Revenue'], ['Q1', '120'], ['Q2', '135'], ['Q3', '150'], ['Q4', '180']]


def bar_chart(width=800, height=520):
    """Titled bar chart with a labelled y axis, gridlines, x labels and value labels, drawn from EXPECTED_TABLE."""
    img = Image.new('RGB', (width, height), 'white')
    d = ImageDraw.Draw(img)
    title_f, tick_f, label_f = ImageFont.load_default(size=24), ImageFont.load_default(size=16), ImageFont.load_default(size=18)
    d.text((width / 2 - d.textlength(CHART_TITLE, font=title_f) / 2, 24), CHART_TITLE, fill='black', font=title_f)
    x0, y0, x1, y1 = 100, 80, width - 40, height - 80
    d.line([(x0, y0), (x0, y1), (x1, y1)], fill='black', width=2)
    ymax = 200
    for v in range(0, ymax + 1, 50):
        y = y1 - (y1 - y0) * v / ymax
        d.line([(x0, y), (x1, y)], fill=(200, 200, 200), width=1)
        d.text((x0 - 12 - d.textlength(str(v), font=tick_f), y - 9), str(v), fill='black', font=tick_f)
    n = len(EXPECTED_TABLE) - 1
    slot = (x1 - x0) / n
    for i, (label, value) in enumerate(EXPECTED_TABLE[1:]):
        bx0 = x0 + slot * i + slot * 0.25
        bx1 = bx0 + slot * 0.5
        by = y1 - (y1 - y0) * int(value) / ymax
        d.rectangle([bx0, by, bx1, y1], fill=(60, 110, 200), outline=(20, 50, 120))
        d.text(((bx0 + bx1) / 2 - d.textlength(label, font=label_f) / 2, y1 + 12), label, fill='black', font=label_f)
        d.text(((bx0 + bx1) / 2 - d.textlength(value, font=tick_f) / 2, by - 22), value, fill='black', font=tick_f)
    d.text((width / 2 - 40, height - 40), 'Quarter', fill='black', font=label_f)
    return img


image = bar_chart()
image_name = 'synthetic_bar_chart_800x520.png'
image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
ceilings = {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PATCHES': MAX_PATCHES, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'MAX_TARGET_TOKENS': MAX_TARGET_TOKENS, 'MAX_TARGET_CHARS': MAX_TARGET_CHARS, 'DECODING': DECODING, 'INSTRUCTION': INSTRUCTION, 'ROW_SEPARATOR': ROW_SEPARATOR, 'CELL_SEPARATOR': CELL_SEPARATOR, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS}
print(ceilings)
input_manifest = validate_inputs(image, max_new_tokens=TABLE_MAX_TOKENS, names=[image_name])
try:
    validate_inputs(image, max_new_tokens=0)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'zero-budget-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/deplot_chart_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'image': image_name, 'rgb_sha256': image_sha256[:16] + '...', 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})
started = time.perf_counter()
result = pipe.extract_table(image, max_new_tokens=TABLE_MAX_TOKENS)
result_seconds = round(time.perf_counter() - started, 3)
print(result['text'])
checks = {
    'text_is_str': isinstance(result['text'], str),
    'budget_respected': result['new_tokens'] <= TABLE_MAX_TOKENS,
    'setting_echoed': result['generation']['max_new_tokens'] == TABLE_MAX_TOKENS and result['generation']['do_sample'] is False,
    'not_truncated': not result['truncated'],
}
if not all(checks.values()):
    raise RuntimeError(f'extract_table output failed a sanity check: {checks}')
frozen_drawn = evaluation_report(result, EXPECTED_TABLE, expected_title=CHART_TITLE, sample_kind='synthetic')
print({'checks': checks, 'seconds': result_seconds, 'new_tokens': result['new_tokens'], 'frozen_drawn_verdict': frozen_drawn['verdict'], 'cell_accuracy': frozen_drawn['metrics'][0]['value'], 'title_match': frozen_drawn['metrics'][1]['value']})

## 6. Baselines and the frozen model on the test charts

Three references frame the adaptation, and none looks at the chart. The **empty baseline** emits no table (the floor: cell accuracy 0). The **header-only baseline** emits, per chart type, the most frequent header row of the training targets and no data rows — what a system that knows only the layout convention scores. The **medoid baseline** emits, per chart type, the training table that best matches the other training tables of its type. The **frozen model** extracts every test chart with the budget from Section 5 and is scored by `pipe.evaluate`: mean relaxed **cell accuracy** (position-wise, so a missing header row or a transposed table shifts every cell), **RNSS** (numbers only, layout-free), exact-table match and the truncation rate, overall and per chart type. SynthChartNet is not DePlot's training family and its targets follow the converter's layout rule, so expect the frozen model's cell accuracy well below its numbers on its own benchmarks; whether it beats the empty table is recorded as `frozen_beats_empty` rather than assumed. The measured values of the first clean run are recorded in `docs/release-verification.md` and the model card.

In [ ]:
baseline_empty = empty_baseline(test_records)
baseline_header = header_only_baseline(train_records, test_records)
baseline_medoid = medoid_baseline(train_records, test_records)
for baseline in (baseline_empty, baseline_header, baseline_medoid):
    print({'baseline': baseline['baseline'], 'cell_accuracy': round(baseline['cell_accuracy'], 3), 'rnss': round(baseline['rnss'], 3), 'note': baseline['note']})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, max_new_tokens=TABLE_MAX_TOKENS)
print({'frozen_model_test': {'cell_accuracy': round(frozen_test['cell_accuracy'], 3), 'rnss': round(frozen_test['rnss'], 3), 'exact_table_match': round(frozen_test['exact_table_match'], 3), 'truncated_rate': round(frozen_test['truncated_rate'], 3)}, 'n': frozen_test['n'], 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'by_chart_type': {'medoid': baseline_medoid['by_chart_type'], 'frozen': frozen_test['by_chart_type']}})
print({'definitions': frozen_test['definitions']})
for row in frozen_test['rows'][:3]:
    print({'chart_type': row['chart_type'], 'target': row['target_text'][:160], 'frozen': row['prediction'][:160], 'cell_accuracy': round(row['cell_accuracy'], 3)})
frozen_beats_empty = frozen_test['cell_accuracy'] > baseline_empty['cell_accuracy']
frozen_beats_medoid = frozen_test['cell_accuracy'] > baseline_medoid['cell_accuracy']
print({'frozen_beats_empty': frozen_beats_empty, 'frozen_beats_medoid': frozen_beats_medoid})

## 7. Bounded fine-tuning of the decoder's last blocks

`pipe.adapt` trains only the last `TRAINABLE_DECODER_LAYERS` blocks of the text decoder plus the decoder's final layer norm — two blocks by default, 18,879,744 of 282,285,696 parameters; the image encoder, every embedding and the untied output projection stay frozen. Each training chart is one sample: the fixed instruction is rendered above the chart exactly as `extract_table` renders it, the frozen encoder reads the composite (recomputed each step without gradients), and the target is the tokenised linearised table (at most `MAX_TARGET_TOKENS` tokens) with its end-of-sequence token, decoded with teacher forcing and scored with the model's own cross-entropy (padding ignored); AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler. Epoch 0 records the frozen model's validation metrics; every epoch is scored on the validation charts and the epoch with the highest validation cell accuracy is kept (ties keep the earlier one). If no epoch beats the frozen model on validation, the selector keeps epoch 0 and the adapter reproduces the frozen tables; that outcome is reported, not hidden.

In [ ]:
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
TRAINABLE_DECODER_LAYERS = 2  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_cell_accuracy': round(entry['val']['cell_accuracy'], 3), 'val_rnss': round(entry['val']['rnss'], 3)})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_decoder_layers=TRAINABLE_DECODER_LAYERS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'training_charts': adapt_result['n_train'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test charts were never used for training or epoch selection, and no test image appears in the training or validation splits. The adapted model is scored exactly as the frozen model was in Section 6, and the five systems — the three baselines, the frozen and the adapted model — are put side by side overall and per chart type. Read it in this order: **cell accuracy** first (the metric the epoch was selected on), then **RNSS** (an adaptation that only learns the target layout raises cell accuracy without reading the numbers any better; RNSS ignores layout), then the per-type split — stacked-bar and line charts are a small share of the test split. `adapted_beats_frozen` records whether held-out cell accuracy rose. A test split of 160 charts from one seeded draw of one shard gives **no dispersion estimate**, so the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on SynthChartNet says nothing about your charts until you measure it there.

In [ ]:
adapted_test = pipe.evaluate(test_records, max_new_tokens=TABLE_MAX_TOKENS)
adapted_val = pipe.evaluate(val_records, max_new_tokens=TABLE_MAX_TOKENS)
comparison = {
    'cell_accuracy': {'empty': round(baseline_empty['cell_accuracy'], 3), 'header_only': round(baseline_header['cell_accuracy'], 3), 'medoid': round(baseline_medoid['cell_accuracy'], 3), 'frozen': round(frozen_test['cell_accuracy'], 3), 'adapted': round(adapted_test['cell_accuracy'], 3)},
    'rnss': {'empty': round(baseline_empty['rnss'], 3), 'header_only': round(baseline_header['rnss'], 3), 'medoid': round(baseline_medoid['rnss'], 3), 'frozen': round(frozen_test['rnss'], 3), 'adapted': round(adapted_test['rnss'], 3)},
    'exact_table_match': {'frozen': round(frozen_test['exact_table_match'], 3), 'adapted': round(adapted_test['exact_table_match'], 3)},
    'delta_vs_frozen': {'cell_accuracy': round(adapted_test['cell_accuracy'] - frozen_test['cell_accuracy'], 3), 'rnss': round(adapted_test['rnss'] - frozen_test['rnss'], 3)},
    'by_chart_type': {chart_type: {'n': row['n'], 'medoid': baseline_medoid['by_chart_type'][chart_type]['cell_accuracy'], 'frozen': row['cell_accuracy'], 'adapted': adapted_test['by_chart_type'][chart_type]['cell_accuracy']} for chart_type, row in frozen_test['by_chart_type'].items()},
}
for key, row in comparison.items():
    print({key: row})
for before, after in list(zip(frozen_test['rows'], adapted_test['rows'], strict=True))[:3]:
    print({'target': before['target_text'][:160], 'frozen': before['prediction'][:160], 'adapted': after['prediction'][:160]})
adapted_beats_frozen = adapted_test['cell_accuracy'] > frozen_test['cell_accuracy']
print({'adapted_beats_frozen': adapted_beats_frozen})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'chart_types': chart_types,
    'max_new_tokens': TABLE_MAX_TOKENS,
    'baselines': {'empty': baseline_empty, 'header_only': baseline_header, 'medoid': baseline_medoid},
    'frozen_test': frozen_test,
    'frozen_beats_empty': frozen_beats_empty,
    'frozen_beats_medoid': frozen_beats_medoid,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
    'adapted_beats_frozen': adapted_beats_frozen,
}
with open('outputs/deplot_chart_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
print({'report': 'outputs/deplot_chart_evaluation_report.json'})

## 9. Re-extract the drawn chart, export the adapter and reload it

The drawn bar chart from Section 5 is extracted again by the adapted model and scored against the table you drew — a different image family from the SynthChartNet charts it was tuned on, and one with a title and a header the SynthChartNet targets never carry, so this is a small look at whether the adaptation changed the model's behaviour *outside* its sample (one chart of evidence, not a measurement; a different table here is a finding to record, not a failure). Both tables are written as CSV.

`pipe.save_artifact` writes the trained tensors — the decoder's last two blocks and its final layer norm, about 76 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `DePlotPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor that is not a decoder tensor, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical tables on eight test charts (VER4).

In [ ]:
import shutil

adapted_result = pipe.extract_table(image, max_new_tokens=TABLE_MAX_TOKENS)
adapted_drawn = evaluation_report(adapted_result, EXPECTED_TABLE, expected_title=CHART_TITLE, sample_kind='synthetic')
print({'frozen': result['text'], 'adapted': adapted_result['text']})
print({'drawn_chart_cell_accuracy': {'frozen': frozen_drawn['metrics'][0]['value'], 'adapted': adapted_drawn['metrics'][0]['value']}})
with open('outputs/deplot_chart_table.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['model', 'row', 'cells'])
    for label, extracted in (('frozen', result), ('adapted', adapted_result)):
        for index, row in enumerate(extracted['table']['rows']):
            writer.writerow([label, index, ' | '.join(row)])

artifact_dir = Path('outputs/deplot_chart_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'deplot_chart', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = DePlotPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [r['text'] for r in pipe.predict(test_records[:8], max_new_tokens=TABLE_MAX_TOKENS)]
after = [r['text'] for r in reloaded.predict(test_records[:8], max_new_tokens=TABLE_MAX_TOKENS)]
parity = {'identical_tables': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_tables'] == parity['of']

weight_entry = next(entry for entry in MANIFEST['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'SafeTensors, loaded in float32, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'release': CORPUS_RELEASE, 'license': CORPUS_LICENSE, 'file': CORPUS_FILE, 'sample_charts': SAMPLE_CHARTS, 'target_rule': 'bar, pie, stacked_bar transposed; line kept; TITLE row first'},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'drawn_chart': {'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'expected_table': EXPECTED_TABLE, 'expected_title': CHART_TITLE}, 'frozen': {k: result[k] for k in ('text', 'new_tokens', 'truncated')}, 'frozen_report': frozen_drawn, 'adapted': {k: adapted_result[k] for k in ('text', 'new_tokens', 'truncated')}, 'adapted_report': adapted_drawn, 'seconds': result_seconds},
    'comparison': comparison,
    'frozen_beats_empty': frozen_beats_empty,
    'frozen_beats_medoid': frozen_beats_medoid,
    'adapted_beats_frozen': adapted_beats_frozen,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/deplot_chart_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen model is a plot-to-table translator scored on charts of a family it was not fine-tuned on, beside three baselines that never look at the chart, and a bounded fine-tuning of the decoder's last two blocks on a few hundred charts is then scored on a held-out test split — overall and per chart type — with an adapter that reloads to identical tables. That is the claim: the adaptation contract works end to end on a real labelled chart corpus, and the numbers it produces are read against the baselines and the frozen model rather than in isolation. Whether held-out cell accuracy rose is recorded as `adapted_beats_frozen`, not assumed.

Cell accuracy is position-wise, so part of any gain can be the model learning the converter's layout — no header row for a single unnamed series, the transposed stacked bars — rather than reading values better; RNSS, which ignores layout, is the check on that. The test split is 160 charts from one seeded draw of one shard, the validation split that picks the epoch is 80, stacked-bar and line charts are a small share of both, and SynthChartNet carries some label noise. **The model generates a table for any image**: an image that is not a chart produces a confident fabrication rather than an empty result, and `truncated` is the only structural flag you get. Fine-tuning on a narrow sample can also erode the model elsewhere; the drawn chart re-extracted in Section 9 is one chart of evidence about that, not a measurement.

Three things to carry to real data. **Baselines first:** the empty, header-only and medoid tables on *your* charts are the numbers to read before any model's, per chart type. **Layout is part of the target:** decide your table orientation and header rule once, from what the frozen model emits, and convert every label to it — a transposed target scores as a miss on every cell. **Leakage:** keep every record of a chart in one split (the contract does this) and split by source document when your charts come from few reports.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled chart corpus, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against three trivial baselines and the frozen model on a held-out split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, accuracy on any other chart population, or production fitness.

**Optional experiments (they do not affect the default path):** raise `LEARNING_RATE` and watch the training loss fall while the validation cell accuracy drops and the selector keeps an early epoch; set `TRAINABLE_DECODER_LAYERS = 1` and compare the artifact size and the held-out scores; remove the value labels from `bar_chart` and see how many cells survive; or bring your own charts through BYOD and read the baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/deplot-chart-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/deplot-chart-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/deplot-chart-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model (Google, Apache-2.0): https://huggingface.co/google/deplot
- Upstream code: https://github.com/google-research/pix2struct
- DePlot: One-shot visual language reasoning by plot-to-table translation (Liu et al., 2022): https://arxiv.org/abs/2212.10505
- Pix2Struct: Screenshot Parsing as Pretraining for Visual Language Understanding (Lee et al., ICML 2023): https://arxiv.org/abs/2210.03347
- ChartQA: A Benchmark for Question Answering about Charts with Visual and Logical Reasoning (Masry et al., 2022): https://arxiv.org/abs/2203.10244
- SynthChartNet (docling-project, CDLA-Permissive-2.0): https://huggingface.co/datasets/docling-project/SynthChartNet
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)